# Results for model: mistralai/mistral-medium-3-instruct

In [ ]:
import polars as pl
import xgboost as xgb
from sklearn.model_selection import train_test_split
import numpy as np

# Load data
train_data = pl.read_parquet('./jane_street_data/train.parquet')

# Feature Engineering
def calculate_top_quantile(df, group_by, feature, target, quantile=0.85):
    return (
        df
        .sort(by=group_by)
        .with_columns(
            pl.col(feature).cast(float),
            pl.col(target).cast(float))
        .group_by(group_by)
        .apply(lambda x: x.with_columns(pl.Series(
            "top_quantile",
            (~x[feature].is_null()).then(x[feature].quantile(quantile)))))
    )

TOP_QUANTILE_df = calculate_top_quantile(train_data, 'date_id', 'feature_00', 'responder_6')

combined_df = train_data.join(TOP_QUANTILE_df[['date_id', 'top_quantile']], on='date_id')
combined_df = combined_df.drop(['id', 'top_quantile'])

# Define target and features
y = combined_df['responder_6'].to_numpy()
X = combined_df.drop(['responder_6']).to_pandas()

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# Remove dates from dataset (avoiding look-ahead bias)
X_train = X_train.drop(columns=['date_id'])
X_test = X_test.drop(columns=['date_id'])

# Define and train model
model = xgb.XGBRegressor(objective='reg:squarederror')
model.fit(X_train, y_train)

# Evaluate model
y_pred = model.predict(X_test)

print("Model trained successfully.")